# Module 02, Notebook 2: Prior Sensitivity Analysis

## Learning Objectives
By completing this notebook, you will:
1. Run a systematic prior sensitivity analysis for ITS causal effect estimates
2. Visualize how prior specification affects the posterior distribution
3. Generate a publication-quality prior sensitivity table
4. Understand when a result is prior-robust vs prior-dependent
5. Apply the prior predictive check workflow

## Why Prior Sensitivity Matters

A Bayesian analysis is only as credible as its priors. The prior predictive check ensures the prior is reasonable before fitting. The sensitivity analysis ensures the conclusion holds across a range of plausible priors. Together, they transform prior specification from a potential weakness into a documented strength.

## Estimated Time: 15 minutes

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az
import causalpy as cp

np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

In [ ]:
# Recreate the smoking ban dataset
np.random.seed(42)
n_months = 48
n_pre = 24
intervention_month = n_pre

months = np.arange(n_months).astype(float)
treated = (months >= intervention_month).astype(float)
t_post = np.maximum(months - intervention_month, 0).astype(float)

true_beta2 = -12.0
true_beta3 = -0.10

ami_rate = (
    85.0 - 0.15 * months + true_beta2 * treated + true_beta3 * t_post * treated
    + np.random.normal(0, 4.0, n_months)
)

df = pd.DataFrame({
    "month": months,
    "ami_rate": ami_rate,
    "treated": treated,
    "t_post": t_post,
})

y_pre_mean = df.loc[df["treated"] == 0, "ami_rate"].mean()
y_pre_std = df.loc[df["treated"] == 0, "ami_rate"].std()

print(f"Pre-intervention: mean={y_pre_mean:.1f}, std={y_pre_std:.1f}")

## 1. Prior Predictive Check

Before fitting the model, sample from the prior and verify the implied outcomes are plausible.

In [ ]:
# Prior predictive check: sample outcomes from the prior before fitting
# This verifies the prior implies plausible outcome values

def run_prior_predictive_check(beta2_sigma_scale, n_samples=100, label=""):
    """Sample from the prior predictive distribution."""
    beta2_sigma = y_pre_std * beta2_sigma_scale

    with pm.Model() as prior_model:
        alpha = pm.Normal("alpha", mu=y_pre_mean, sigma=2 * y_pre_std)
        beta1 = pm.Normal("beta1", mu=0, sigma=y_pre_std * 0.1)
        beta2 = pm.Normal("beta2", mu=0, sigma=beta2_sigma)  # Varying this
        beta3 = pm.Normal("beta3", mu=0, sigma=y_pre_std * 0.1)
        sigma = pm.HalfNormal("sigma", sigma=y_pre_std)

        mu = alpha + beta1 * months + beta2 * treated + beta3 * t_post
        y_prior = pm.Normal("y_prior", mu=mu, sigma=sigma)

        prior_pred = pm.sample_prior_predictive(samples=n_samples, random_seed=42)

    return prior_pred.prior_predictive["y_prior"].values.squeeze()


# Run prior predictive checks for three prior widths
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

prior_configs = [
    (0.3, "Tight prior: β₂ ~ N(0, 0.3σ_Y)"),
    (1.0, "Default prior: β₂ ~ N(0, σ_Y)"),
    (5.0, "Diffuse prior: β₂ ~ N(0, 5σ_Y)"),
]

for ax, (scale, label) in zip(axes, prior_configs):
    prior_samples = run_prior_predictive_check(scale)

    for i in range(min(50, prior_samples.shape[0])):
        ax.plot(months, prior_samples[i], alpha=0.12, color="#3498db", linewidth=0.7)

    # Plot observed data for reference
    ax.scatter(months, ami_rate, s=15, color="#e74c3c", zorder=5, alpha=0.7)
    ax.axvline(intervention_month, color="black", linestyle="--", linewidth=1.5)

    # Show plausible range (0 to 150 for AMI rates)
    ax.set_ylim(-50, 250)
    ax.axhline(0, color="gray", linestyle=":", alpha=0.5, label="Zero baseline")

    ax.set_title(label, fontsize=10)
    ax.set_xlabel("Month")

axes[0].set_ylabel("AMI Rate per 100,000")
plt.suptitle("Prior Predictive Check: Blue Lines = Prior-Implied Outcomes, Red Dots = Observed",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

print("Interpretation:")
print("  Tight prior: Trajectories cluster near observed data (strong regularization)")
print("  Default prior: Spans a reasonable range of plausible AMI rates")
print("  Diffuse prior: Trajectories go to extreme values (including negative, unrealistic)")

## 2. Fit Models with Different Priors

In [ ]:
# Fit the ITS model with three different prior widths for beta_2
# Compare posterior estimates for the level change

def fit_its_with_prior(beta2_sigma_scale, label):
    """Fit ITS model with a specific prior on the level change."""
    beta2_sigma = y_pre_std * beta2_sigma_scale

    with pm.Model() as model:
        alpha = pm.Normal("Intercept", mu=y_pre_mean, sigma=2 * y_pre_std)
        beta1 = pm.Normal("month", mu=0, sigma=y_pre_std * 0.1)
        beta2 = pm.Normal("treated", mu=0, sigma=beta2_sigma)
        beta3 = pm.Normal("t_post", mu=0, sigma=y_pre_std * 0.1)
        sigma = pm.HalfNormal("sigma", sigma=y_pre_std)

        mu = alpha + beta1 * months + beta2 * treated + beta3 * t_post
        y_hat = pm.Normal("y_hat", mu=mu, sigma=sigma, observed=ami_rate)

        idata = pm.sample(
            draws=1000, tune=1000, chains=4,
            progressbar=False, random_seed=42,
        )

    return idata, label, beta2_sigma_scale


print("Fitting three models (this may take a few minutes)...")

tight_idata, tight_label, _ = fit_its_with_prior(0.3, "Tight (0.3σ_Y)")
print("  Tight prior model: done")

default_idata, default_label, _ = fit_its_with_prior(1.0, "Default (1.0σ_Y)")
print("  Default prior model: done")

diffuse_idata, diffuse_label, _ = fit_its_with_prior(5.0, "Diffuse (5.0σ_Y)")
print("  Diffuse prior model: done")

print("All models fitted.")

## 3. Compare Posterior Distributions

In [ ]:
# Forest plot comparing posteriors across three prior specifications

az.plot_forest(
    [tight_idata, default_idata, diffuse_idata],
    model_names=["Tight (0.3σ_Y)", "Default (1.0σ_Y)", "Diffuse (5.0σ_Y)"],
    var_names=["treated", "t_post"],
    combined=True,
    hdi_prob=0.94,
    figsize=(10, 4),
)
plt.axvline(0, color="red", linestyle="--", linewidth=1.5, alpha=0.7)
plt.axvline(true_beta2, color="#27ae60", linestyle=":", linewidth=2, label=f"True β₂={true_beta2}")
plt.title("Prior Sensitivity: Posterior Distributions for Three Prior Widths", fontsize=13)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Generate a structured prior sensitivity table
# This is the format used in academic publications

models = [
    (tight_idata, "Tight: N(0, 0.3σ_Y)"),
    (default_idata, "Default: N(0, 1.0σ_Y)"),
    (diffuse_idata, "Diffuse: N(0, 5.0σ_Y)"),
]

print("Prior Sensitivity Analysis: Smoking Ban → AMI Rate")
print("=" * 75)
print(f"{'Prior Specification':<30} {'β₂ Mean':>10} {'β₂ HDI (94%)':>22} {'P(β₂<0)':>10}")
print("-" * 75)

for idata, label in models:
    samples = idata.posterior["treated"].values.flatten()
    mean = samples.mean()
    hdi = az.hdi(samples, hdi_prob=0.94)
    prob_neg = (samples < 0).mean()
    print(f"  {label:<28} {mean:>+10.2f} [{hdi[0]:>+8.2f}, {hdi[1]:>+8.2f}] {prob_neg:>10.1%}")

print("-" * 75)
print(f"  True value:                    {true_beta2:>+10.2f}")
print()
print("CONCLUSION:")

# Check if all three agree on direction
probs = []
for idata, _ in models:
    samples = idata.posterior["treated"].values.flatten()
    probs.append((samples < 0).mean())

if all(p > 0.90 for p in probs):
    print("  PRIOR-ROBUST: All three prior specifications show >90% P(effect < 0).")
    print("  The negative effect of the smoking ban is supported across all prior widths.")
else:
    print("  PRIOR-SENSITIVE: Results depend on prior specification.")
    print("  Additional data or stronger prior justification is needed.")

In [ ]:
# Overlay posterior distributions for visual comparison

fig, ax = plt.subplots(figsize=(10, 5))

colors = ["#95a5a6", "#3498db", "#e74c3c"]
labels = ["Tight (0.3σ_Y)", "Default (1.0σ_Y)", "Diffuse (5.0σ_Y)"]

for (idata, _), color, label in zip(models, colors, labels):
    samples = idata.posterior["treated"].values.flatten()
    az.plot_posterior(
        samples,
        hdi_prob=0.94,
        color=color,
        label=label,
        point_estimate=None,
        ax=ax,
    )

ax.axvline(true_beta2, color="#27ae60", linestyle=":", linewidth=2.5,
           label=f"True β₂ = {true_beta2}")
ax.axvline(0, color="black", linestyle="--", linewidth=1.5)
ax.set_title("Prior Sensitivity: Posterior Distributions for Level Change (β₂)", fontsize=13)
ax.set_xlabel("Level Change (AMI per 100k)", fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## Summary

### Key Findings

1. **Prior predictive check** reveals that the diffuse prior allows unrealistic trajectories (negative AMI rates, effects larger than the full outcome range). The default prior spans a reasonable range.

2. **Prior sensitivity analysis** shows that all three prior specifications give essentially the same posterior for the level change β₂. The data are informative enough to dominate the prior — this is a prior-robust result.

3. The tight prior produces slightly narrower credible intervals (prior is regularizing toward zero), but the posterior mean is similar across all three specifications.

### When Results Are Not Prior-Robust

If the posterior changes substantially between tight and diffuse priors:
- The data are underpowered (short post-period, high noise)
- This is an honest signal: more data are needed for a robust conclusion
- Report the sensitivity analysis — do not hide the prior dependence
- Consider whether a meta-analytic (informative) prior is justified

### What's Next
**Notebook 3:** Posterior predictive checks — verifying the model fits the data well after sampling.